# Air Quality Prediction System — Phase 1 & 2
This notebook performs Phase 1 (Load) and Phase 2 (Inspect & initial preprocessing) for the project 'A Comparative Analysis of Time-Series Models for Real-time Air Quality Prediction'.

Instructions:
1. Run the environment setup commands (once) in your PowerShell/Anaconda prompt.
2. Run the Python cells below to load `feeds.csv`.
3. Copy and paste the entire output of `df.info()` back into the chat so I can produce the next tailored preprocessing steps for Phase 2 and then move to Phase 3 (EDA).

## Environment setup (Conda) — run once in PowerShell / Anaconda Prompt
Run the following to create a clean environment named `airquality_env` and install the essential libraries we need for the project:

```powershell
conda create -n airquality_env python=3.10 jupyter pandas numpy matplotlib seaborn scikit-learn statsmodels -y
conda activate airquality_env
# (Optional) Install a few extra ML libraries via pip inside the activated env:
pip install xgboost lightgbm tensorflow
```
If you prefer pip-only (not recommended for base Windows), here's an alternative:
```powershell
pip install pandas numpy matplotlib seaborn scikit-learn statsmodels xgboost lightgbm tensorflow jupyter
```

## Phase 1: Load Data
The code below tries to load `feeds.csv`. It attempts both Excel and CSV readers where appropriate and fails fast if the file is missing. After loading it prints `df.info()`, `df.head(10)`, `df.describe()`, and missing value counts.

In [15]:
import pandas as pd
import numpy as np
import os

FILE_NAME = 'feeds.csv'
print(f"--- [Phase 1] Loading data from '{FILE_NAME}' ---")

# try read_excel first (in case the file is actually an xlsx) then fallback to read_csv
df = None
# check file exists in working dir
if not os.path.exists(FILE_NAME):
    raise FileNotFoundError(f"File '{FILE_NAME}' not found in working directory: {os.getcwd()}")

# attempt both readers robustly
last_err = None
try:
    df = pd.read_excel(FILE_NAME)
    print(f"Loaded with pd.read_excel. Shape: {df.shape}")
except Exception as e_excel:
    last_err = e_excel
    try:
        df = pd.read_csv(FILE_NAME)
        print(f"Loaded with pd.read_csv. Shape: {df.shape}")
    except Exception as e_csv:
        # raise the last error with context
        print('Failed to read file with both read_excel and read_csv')
        raise

# --- Phase 2: Initial Inspection ---

print('[INFO] DataFrame Info:')
# df.info() prints directly; in notebooks it will render below
df.info()
print('\n[HEAD] First 10 Rows:')
print(df.head(10))
print('\n[DESCRIBE] Statistical Summary:')
print(df.describe(include='all'))
print('\n[NULL COUNT] Missing values per column:')
print(df.isnull().sum())

--- [Phase 1] Loading data from 'feeds.csv' ---


C:\Users\burak\AppData\Local\Temp\ipykernel_17644\1991856471.py:22: DtypeWarning: Columns (9) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(FILE_NAME)


Loaded with pd.read_csv. Shape: (291472, 10)
[INFO] DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 291472 entries, 0 to 291471
Data columns (total 10 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   created_at  291472 non-null  object 
 1   entry_id    291472 non-null  int64  
 2   field1      291309 non-null  float64
 3   field2      291309 non-null  float64
 4   field3      291309 non-null  float64
 5   field4      291309 non-null  float64
 6   latitude    0 non-null       float64
 7   longitude   0 non-null       float64
 8   elevation   0 non-null       float64
 9   status      163 non-null     object 
dtypes: float64(7), int64(1), object(2)
memory usage: 22.2+ MB

[HEAD] First 10 Rows:
                  created_at  entry_id    field1    field2  field3  field4  \
0  2025-03-05T03:37:33+00:00         1  31.81915  33.39748    24.5    69.7   
1  2025-03-05T03:37:53+00:00         2  31.84471  33.19578    18.6    48.8 

## Phase 2: Preprocessing (auto-detect timestamp & set index)
Run the cell below immediately after the previous cell. It will: detect the most likely timestamp column, convert it to datetime, set it as the DataFrame index (sorted), and print missing value counts.

Note: I intentionally attempt automatic detection so you don't need to provide the exact timestamp column name yet — but please still copy the full output of `df.info()` back here so I can confirm and tailor later steps.

In [16]:
# Auto-detect timestamp-like column, convert to datetime, set index and report missing values
from collections import Counter

def find_datetime_column(df):
    # Candidate by name tokens
    name_tokens = ['date', 'time', 'timestamp', 'created', 'datetime']
    candidates = [c for c in df.columns if any(tok in c.lower() for tok in name_tokens)]
    best_col = None
    best_count = -1
    # Prioritize name-based candidates
    if candidates:
        for c in candidates:
            parsed = pd.to_datetime(df[c], errors='coerce', infer_datetime_format=True)
            count = parsed.notna().sum()
            if count > best_count:
                best_count = count
                best_col = c
        # if more than half of values parsed, accept
        if best_count >= 0.5 * len(df):
            return best_col
    # Fallback: try every column and pick the one with most parseable datetimes
    for c in df.columns:
        try:
            parsed = pd.to_datetime(df[c], errors='coerce', infer_datetime_format=True)
            count = parsed.notna().sum()
            if count > best_count:
                best_count = count
                best_col = c
        except Exception:
            continue
    # require at least some parseable values (e.g., 10% of rows) to consider it a datetime column
    if best_count >= max(3, 0.1 * len(df)):
        return best_col
    return None

# Find timestamp column
ts_col = find_datetime_column(df)
print(f"Auto-detected timestamp column: {ts_col}")

if ts_col is None:
    raise ValueError('No suitable datetime-like column could be auto-detected. Please inspect `df.info()` output and tell me the timestamp column name.')

# Convert to datetime (coerce invalid -> NaT)
df[ts_col] = pd.to_datetime(df[ts_col], errors='coerce', infer_datetime_format=True)
n_nat = df[ts_col].isna().sum()
print(f"{n_nat} rows could not be converted to datetime (NaT) in column '{ts_col}'.")

# Set as index and sort
df = df.set_index(ts_col)
df = df.sort_index()

# Report index frequency guess and missing values
try:
    freq = pd.infer_freq(df.index[:min(len(df.index), 1000)])
except Exception:
    freq = None
print(f"Inferred index frequency (first 1000 rows): {freq}")

print('\nMissing values per column after setting datetime index:')
print(df.isnull().sum())

# Display a small slice for verification
print('\nData sample (first 5 rows after indexing):')
print(df.head())

# Keep df in global namespace for downstream cells
globals()['df'] = df

C:\Users\burak\AppData\Local\Temp\ipykernel_17644\2505206579.py:13: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  parsed = pd.to_datetime(df[c], errors='coerce', infer_datetime_format=True)
C:\Users\burak\AppData\Local\Temp\ipykernel_17644\2505206579.py:44: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  df[ts_col] = pd.to_datetime(df[ts_col], errors='coerce', infer_datetime_format=True)


Auto-detected timestamp column: created_at
0 rows could not be converted to datetime (NaT) in column 'created_at'.
Inferred index frequency (first 1000 rows): None

Missing values per column after setting datetime index:
entry_id          0
field1          163
field2          163
field3          163
field4          163
latitude     291472
longitude    291472
elevation    291472
status       291309
dtype: int64

Data sample (first 5 rows after indexing):
                           entry_id    field1    field2  field3  field4  \
created_at                                                                
2025-03-05 03:37:33+00:00         1  31.81915  33.39748    24.5    69.7   
2025-03-05 03:37:53+00:00         2  31.84471  33.19578    18.6    48.8   
2025-03-05 03:38:13+00:00         3  31.88725  34.05800    17.7    42.3   
2025-03-05 03:38:33+00:00         4  31.93970  34.77564    16.1    46.3   
2025-03-05 03:38:53+00:00         5  31.96106  33.70981    15.4    44.3   

                

In [ ]:
# === Phase 2 (explicit) Preprocessing based on df.info() output ===
# Assumptions from df.info(): timestamp column is 'created_at' and sensors are field1..field4.
# Mapping assumption: field1->pm2_5, field2->pm10, field3->temperature, field4->humidity.
import numpy as np

# Assume datetime index is already set by previous cell
print("Assuming datetime index is already set from previous cell.")

# Rename sensor columns to canonical names (adjust mapping if different)
rename_map = {'field1':'pm2_5','field2':'pm10','field3':'temperature','field4':'humidity'}
existing_renames = {k:v for k,v in rename_map.items() if k in df.columns}
df = df.rename(columns=existing_renames)
print('Columns after rename:', df.columns.tolist())

# Report missing values
print('\nMissing values before cleaning:')
print(df.isnull().sum())

# Drop columns that are entirely null (e.g., latitude/longitude/elevation)
cols_all_null = [c for c in df.columns if df[c].isnull().all()]
if cols_all_null:
    print('Dropping all-null columns:', cols_all_null)
    df = df.drop(columns=cols_all_null)

# Drop entry_id (redundant when timestamp exists)
if 'entry_id' in df.columns:
    df = df.drop(columns=['entry_id'])
    print("Dropped column: entry_id")

# Ensure sensor columns are numeric
for c in ['pm2_5','pm10','temperature','humidity']:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors='coerce')

# Handle impossible/outlier values: set to NaN so interpolation can fill them
if 'pm2_5' in df.columns:
    bad = (df['pm2_5'] < 0) | (df['pm2_5'] > 5000)
    print(f"pm2_5 impossible values: {bad.sum()} -> set to NaN")
    df.loc[bad, 'pm2_5'] = np.nan
if 'pm10' in df.columns:
    bad = (df['pm10'] < 0) | (df['pm10'] > 5000)
    print(f"pm10 impossible values: {bad.sum()} -> set to NaN")
    df.loc[bad, 'pm10'] = np.nan
if 'humidity' in df.columns:
    bad = (df['humidity'] < 0) | (df['humidity'] > 100)
    print(f"humidity impossible values: {bad.sum()} -> set to NaN")
    df.loc[bad, 'humidity'] = np.nan
if 'temperature' in df.columns:
    bad = (df['temperature'] < -50) | (df['temperature'] > 60)
    print(f"temperature impossible values: {bad.sum()} -> set to NaN")
    df.loc[bad, 'temperature'] = np.nan

print('\nMissing values after basic cleaning:')
print(df.isnull().sum())

# Keep df in namespace for downstream cells and optionally save a checkpoint
globals()['df'] = df
# Optionally: df.to_csv('feeds_preprocessed_step1.csv')

Datetime index already set. Skipping timestamp conversion.
Columns after rename: ['entry_id', 'pm2_5', 'pm10', 'temperature', 'humidity', 'latitude', 'longitude', 'elevation', 'status']

Missing values before cleaning:
entry_id            0
pm2_5             163
pm10              163
temperature       163
humidity          163
latitude       291472
longitude      291472
elevation      291472
status         291309
dtype: int64
Dropping all-null columns: ['latitude', 'longitude', 'elevation']
Dropped column: entry_id
pm2_5 impossible values: 0 -> set to NaN
pm10 impossible values: 0 -> set to NaN
humidity impossible values: 1056 -> set to NaN
temperature impossible values: 1168 -> set to NaN

Missing values after basic cleaning:
pm2_5             163
pm10              163
temperature      1331
humidity         1219
status         291309
dtype: int64


## Remaining Phase 2 TODOs
The rest of Phase 2 will be performed after you run the cells above and send me `df.info()` output (so I can confirm columns). Tentative tasks (I'll execute these once you confirm):

- Identify and rename sensor columns to canonical names: `pm2_5`, `pm10`, `temperature`, `humidity`.
- Drop useless columns (e.g., `entry_id`) if present.
- Handle missing data: report gap sizes; apply time-based interpolation (linear/time) or forward-fill for short gaps.
- Handle outliers: clip/remove physically impossible values (PM < 0, humidity not in [0,100], temperature extremes).
- Resample to consistent frequency (e.g., hourly `H`) using `.resample('H').mean()` and report coverage.
- Save cleaned dataset to `feeds_cleaned.csv` for Phase 3.

When you run the previous cells and paste the full `df.info()` output here, I will produce the exact code for renaming and the missing-data strategy.

## Phase 2: Preprocessing (continued) — Aggressive Time-Based Missing Value Correction
This cell performs aggressive filling of missing values using rolling window averages, handles any remaining sentinels, resamples to hourly frequency, and saves the cleaned data.

It assumes the DataFrame `df` is already loaded and the initial preprocessing is done (timestamp set as index, columns renamed, etc.).

Key steps:
1. **Sentinel Handling**: Treats any remaining implausible values (e.g., PM > 500, humidity > 100, temperature > 50) as NaN.
2. **Rolling Window Filling**: Fills NaNs using rolling mean over a window (e.g., 10 samples), then applies forward/backward fill as fallback.
3. **Hourly Resampling**: Resamples the data to hourly frequency using mean aggregation.
4. **Save Cleaned Data**: Outputs the cleaned and resampled data to `feeds_cleaned.csv`.

In [18]:
# === Phase 2 (continued): Aggressive Time-Based Missing Value Correction ===
# Use rolling window averages to fill missing values robustly for time-series data.
# This is aggressive: fills gaps with local averages over a window (e.g., 5 samples before/after).
# After filling, resample to hourly frequency using mean aggregation.

import numpy as np

print("Starting aggressive missing value correction using rolling window averages...")

# Define sensor columns
sensor_cols = ['pm2_5', 'pm10', 'temperature', 'humidity']

# First, handle any remaining sentinel/outlier values (e.g., very high values as NaN)
# From describe, max field3=999.9, field4=1999.9 — treat as sentinels
for col in sensor_cols:
    if col in df.columns:
        # Assume sentinels are > 100 for PM, > 100 for humidity, > 50 for temp (adjust if needed)
        if 'pm' in col:
            sentinel_threshold = 500  # PM values >500 are unlikely
        elif col == 'humidity':
            sentinel_threshold = 100
        elif col == 'temperature':
            sentinel_threshold = 50
        else:
            sentinel_threshold = np.inf  # no threshold
        bad = df[col] > sentinel_threshold
        if bad.any():
            print(f"Setting {bad.sum()} sentinel values in {col} to NaN")
            df.loc[bad, col] = np.nan

print(f"Missing values before filling: {df[sensor_cols].isnull().sum().sum()} total")

# Aggressive filling: use rolling mean over a window (e.g., 10 samples) to fill NaNs
# This propagates local averages forward/backward
window_size = 10  # Adjust based on data frequency; for ~20s intervals, 10 samples ~3-5 min window
for col in sensor_cols:
    if col in df.columns:
        # Rolling mean, then fill NaNs with the rolling mean
        rolling_mean = df[col].rolling(window=window_size, center=True, min_periods=1).mean()
        df[col] = df[col].fillna(rolling_mean)
        # If still NaNs (e.g., at edges), use forward/backward fill as fallback
        df[col] = df[col].fillna(method='ffill').fillna(method='bfill')

print(f"Missing values after rolling window filling: {df[sensor_cols].isnull().sum().sum()} total")

# Report final missing counts
print("\nFinal missing values per sensor column:")
print(df[sensor_cols].isnull().sum())

# Now resample to hourly frequency ('H') using mean
print("\nResampling to hourly frequency using mean aggregation...")
df_hourly = df[sensor_cols].resample('H').mean()
print(f"Resampled shape: {df_hourly.shape}")
print(f"Hourly data range: {df_hourly.index.min()} to {df_hourly.index.max()}")

# Report coverage: percent of hours with at least one non-NaN value
coverage = df_hourly.notna().any(axis=1).sum() / len(df_hourly) * 100
print(f"Hourly coverage (hours with any data): {coverage:.1f}%")

# Save cleaned dataset
output_file = 'feeds_cleaned.csv'
df_hourly.to_csv(output_file)
print(f"\nSaved cleaned hourly dataset to '{output_file}'")

# Keep in namespace
globals()['df_hourly'] = df_hourly

Starting aggressive missing value correction using rolling window averages...
Setting 2736 sentinel values in temperature to NaN
Missing values before filling: 5612 total
Missing values after rolling window filling: 0 total

Final missing values per sensor column:
pm2_5          0
pm10           0
temperature    0
humidity       0
dtype: int64

Resampling to hourly frequency using mean aggregation...
Resampled shape: (3531, 4)
Hourly data range: 2025-03-05 03:00:00+00:00 to 2025-07-30 05:00:00+00:00
Hourly coverage (hours with any data): 66.7%

Saved cleaned hourly dataset to 'feeds_cleaned.csv'
Missing values after rolling window filling: 0 total

Final missing values per sensor column:
pm2_5          0
pm10           0
temperature    0
humidity       0
dtype: int64

Resampling to hourly frequency using mean aggregation...
Resampled shape: (3531, 4)
Hourly data range: 2025-03-05 03:00:00+00:00 to 2025-07-30 05:00:00+00:00
Hourly coverage (hours with any data): 66.7%

Saved cleaned hou

C:\Users\burak\AppData\Local\Temp\ipykernel_17644\1582162889.py:42: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df[col] = df[col].fillna(method='ffill').fillna(method='bfill')
C:\Users\burak\AppData\Local\Temp\ipykernel_17644\1582162889.py:52: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df_hourly = df[sensor_cols].resample('H').mean()


# === Project Phase 1 & 2 Complete ===
# Summary of completed work:
# - Loaded feeds.csv (291,472 rows, 10 columns from ThingSpeak)
# - Inspected data: timestamp 'created_at', sensors field1-4, all geo columns null
# - Auto-detected and set 'created_at' as datetime index (UTC)
# - Renamed sensors: field1->pm2_5, field2->pm10, field3->temperature, field4->humidity
# - Dropped null columns (latitude/longitude/elevation) and entry_id
# - Marked 3904 temperature sentinels (>50) and 1056 humidity sentinels (>100) as NaN
# - Applied aggressive rolling window filling (window=10) + ffill/bfill for missing values
# - Resampled to hourly frequency using mean aggregation
# - Result: 3531 hourly rows, 66.7% coverage, saved to 'feeds_cleaned.csv'
#
# Next: Phase 3 EDA - Load 'feeds_cleaned.csv' and perform exploratory data analysis:
# - Time series plots for each sensor
# - Correlation matrix and heatmaps
# - Seasonal decomposition (daily/weekly patterns)
# - Outlier detection and distribution analysis
# - Stationarity tests for time-series modeling preparation